# 03 — URL preprocessing và feature engineering
**Phụ trách:** Nguyễn Khang Hy  
**Trạng thái:** đã triển khai, chờ review.

**Đầu ra:** parser, domain đăng ký, bộ features có định nghĩa và ma trận X gắn id. Tập 21 ứng viên là tham khảo schema, không phải bộ tối ưu đã xác nhận.

## 0. Môi trường và đường dẫn
Mặc định chạy local: cài requirements.txt trong .venv, mở notebook bằng VS Code/Jupyter và chọn kernel của .venv. Dataset ở data/ hoặc vị trí PHISHGUARD_DATA_DIR trên máy mình. Ô dưới chỉ nạp đường dẫn/config; Colab là lựa chọn thêm theo README.

In [1]:
from pathlib import Path
import os
import sys
import tomllib

start = Path(os.environ.get("PHISHGUARD_ROOT", Path.cwd())).expanduser().resolve()
candidates = (start, *start.parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / "pyproject.toml").is_file()
     and (p / "src" / "phishguard_vn").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Chuyen working directory vao repo hoac dat PHISHGUARD_ROOT "
        "toi thu muc du an tren may ban."
    )
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from phishguard_vn.paths import get_data_dir, get_config_path
DATA_DIR = get_data_dir()
with get_config_path().open("rb") as handle:
    CONFIG = tomllib.load(handle)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Config:", CONFIG["experiment"]["name"])


Project: /Users/khangcero/PhishGuardVN
Dataset: /Users/khangcero/PhishGuardVN/data
Config: week4_e1_core_lr


## 1. Parser và quy ước chuẩn hóa
`url_parser.parse_url` giữ `raw_url`; thêm `http://` khi thiếu scheme (URL bắt đầu bằng `//` thì thêm `http:`). Host được chuẩn hóa trước khi tạo khóa nhóm: hạ chữ thường, bỏ dấu chấm cuối, IDN → Punycode, IP → dạng rút gọn chuẩn. Domain đăng ký tách bằng `tldextract` 5.4.0 với snapshot PSL đóng gói sẵn (không tải mạng), `include_psl_private_domains=False` (ví dụ `*.vercel.app` gộp thành `vercel.app`). Path/query giữ chữ hoa/thường; dùng `urlsplit` nên giữ `;params`. IP hợp lệ nhưng không có domain (`is_ip_no_domain`). Mã lỗi: `missing_url`, `empty_url`, `invalid_port`, `invalid_hostname`, `unresolved_domain`, `parsing_error:<Loại>`. Dòng lỗi không vào X mà ghi ở `03_excluded_rows.csv`.

In [2]:
from phishguard_vn.url_parser import parse_url
from phishguard_vn.features import CANDIDATE_FEATURES, extract_features, build_feature_table

print("Reference candidate count:", len(CANDIDATE_FEATURES))

# Test xem parser chạy đúng không
test_url = "https://login-bank.com/verify?id=123"
parsed = parse_url(test_url)

print("Kết quả bóc tách URL:")
print(parsed)


Reference candidate count: 12
Kết quả bóc tách URL:
ParsedURL(raw_url='https://login-bank.com/verify?id=123', hostname='login-bank.com', registrable_domain='login-bank.com', subdomain='', path='/verify', query='id=123', port=None, scheme_was_added=False, is_ip=False, is_valid=True, error_reason=None)


## 2. Định nghĩa và trích xuất features
Giữ 12/21 lexical features (số đếm và độ dài). Bỏ `special_cnt` (dư thừa tuyến tính: url_len = letter + digit + special), các feature tỷ lệ (suy ra được từ cột đã giữ) và các feature query (`?`, `=`, `&`, query_len) vì dữ liệu gần như không có query (0,08% URL trong url_train.csv). `dom_len` tính theo registrable domain, khớp vn_compphish.csv. Loại `is_https` và metadata khỏi X. `is_ip` là cột hằng trên dữ liệu hiện tại (0 URL dạng IPv4) nhưng giữ để schema nhất quán. Định nghĩa đầy đủ ở feature_dictionary.csv; cột `train_check` còn các ô chưa đo.

In [3]:
from phishguard_vn.features import CANDIDATE_FEATURES, extract_features

print("Danh sách 12 Features đã chốt:", CANDIDATE_FEATURES)

# Test thử tính toán đặc trưng
feats = extract_features("https://login-bank.com/verify?id=123")
print("\nKết quả trích xuất Features:")
print(feats)

Danh sách 12 Features đã chốt: ('url_len', 'dom_len', 'is_ip', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'slash_cnt', 'entropy', 'path_len')

Kết quả trích xuất Features:
{'url_len': 28.0, 'dom_len': 14.0, 'is_ip': 0.0, 'subdom_cnt': 0.0, 'letter_cnt': 20.0, 'digit_cnt': 3.0, 'dot_cnt': 1.0, 'dash_cnt': 1.0, 'under_cnt': 0.0, 'slash_cnt': 1.0, 'entropy': 4.494680368408909, 'path_len': 7.0}


## 3. Kiểm tra các trường hợp dễ sai
14 test case gồm thiếu scheme, domain nhiều cấp, IP v4/v6, @, port lỗi, Punycode, hoa/thường, đầu vào rỗng/None, URL bắt đầu bằng `//`; cộng thêm các cặp host tương đương (IDN, IPv6, hoa/thường, dấu chấm cuối) phải cùng khóa nhóm, và path giữ nguyên `;params`.

In [ ]:
from phishguard_vn.url_parser import parse_url

# (url, registrable_domain, is_valid, is_ip, error_reason)
CASES = [
    ("google.com",                         "google.com",       True,  False, None),
    ("HTTPS://WWW.Example.COM/Path?Q=A",   "example.com",      True,  False, None),
    ("a.b.c.example.co.uk/x",              "example.co.uk",    True,  False, None),
    ("example.com:8080/a",                 "example.com",      True,  False, None),
    ("http://paypal.com@evil.com/login",   "evil.com",         True,  False, None),
    ("https://xn--mnchen-3ya.de/",         "xn--mnchen-3ya.de", True, False, None),
    ("http://192.168.1.1/login",           None,               True,  True,  "is_ip_no_domain"),
    ("http://[2001:db8::1]/",              None,               True,  True,  "is_ip_no_domain"),
    ("https://example.com:99999/",         None,               False, False, "invalid_port"),
    ("invalid_url",                        None,               False, False, "unresolved_domain"),
    ("",                                   None,               False, False, "empty_url"),
    (None,                                 None,               False, False, "missing_url"),
    ("//example.com/Path",                 "example.com",      True,  False, None),
    ("https://example.com:bad/p",          None,               False, False, "invalid_port"),
]

print("--- CHẠY ASSERTIONS KIỂM THỬ ---")
for url, exp_domain, exp_valid, exp_ip, exp_err in CASES:
    p = parse_url(url)
    got = (p.registrable_domain, p.is_valid, p.is_ip, p.error_reason)
    exp = (exp_domain, exp_valid, exp_ip, exp_err)
    assert got == exp, f"FAIL {url!r}: got={got} expected={exp}"

# IP không bao giờ được làm group key
assert all(parse_url(u).registrable_domain is None for u in ("http://192.168.1.1/", "http://[2001:db8::1]/"))

# Giữ case của path/query, hạ chữ thường hostname
p = parse_url("HTTPS://WWW.Example.COM/Path?Q=A")
assert p.path == "/Path" and p.query == "Q=A"

# Cặp tương đương phải cùng khóa nhóm
for a, b in [("http://bücher.de/", "http://xn--bcher-kva.de/"),
             ("http://[2001:db8::1]/", "http://[2001:0db8:0:0:0:0:0:1]/"),
             ("http://EXAMPLE.com./", "example.com")]:
    pa, pb = parse_url(a), parse_url(b)
    assert (pa.registrable_domain, pa.hostname) == (pb.registrable_domain, pb.hostname), (a, b)

# Path giữ nguyên ';params' và URL bắt đầu bằng //
assert parse_url("https://example.com/a;b?x=1").path == "/a;b"
assert parse_url("//example.com/Path").path == "/Path"

print(f"Tất cả {len(CASES)} test cases pass!")

--- CHẠY ASSERTIONS KIỂM THỬ ---
Tất cả 12 test cases pass!


## 4. Ma trận features và bàn giao
X giữ id + 12 features, chỉ gồm các dòng parse hợp lệ. Dòng lỗi được ghi riêng ở `03_excluded_rows.csv` (id, error_reason, url). Kiểm tra ở cell handoff bên dưới.

In [ ]:
import numpy as np
import pandas as pd
from phishguard_vn.features import build_feature_table, CANDIDATE_FEATURES

ID_COL = "id"
FEATURE_COLS = list(CANDIDATE_FEATURES)
assert len(FEATURE_COLS) == 12

input_path = PROJECT_ROOT / "data" / "interim" / "core_gold_silver.csv"
frame = pd.read_csv(input_path)
assert frame[ID_COL].notna().all(), "Có id thiếu trong dữ liệu đầu vào"

print(f"Đang trích xuất features cho {len(frame)} dòng...")
feature_table = build_feature_table(frame)

out_dir = PROJECT_ROOT / "data" / "interim"
out_dir.mkdir(parents=True, exist_ok=True)

ok = feature_table["is_valid"].astype(bool)

# Metadata cho Hưng: tất cả các dòng
domain_meta = feature_table[[ID_COL, "registrable_domain", "error_reason"]].copy()
domain_meta.to_csv(out_dir / "03_registrable_domains.csv", index=False)

# Ma trận X cho Phúc: chỉ dòng parse hợp lệ, chỉ id + 12 features
matrix_X = feature_table.loc[ok, [ID_COL, *FEATURE_COLS]].copy()
assert list(matrix_X.columns) == [ID_COL, *FEATURE_COLS], "Sai cột/thứ tự của X"
assert not matrix_X[FEATURE_COLS].isna().any().any(), "X còn NaN"
matrix_X.to_csv(out_dir / "03_matrix_X_features.csv", index=False)

# Bảng id bị lỗi và lý do (bàn giao cho Khiêm theo mục 02 phân công)
excluded = (feature_table.loc[~ok, [ID_COL, "error_reason"]]
            .merge(frame[[ID_COL, "url"]], on=ID_COL, how="left"))
excluded.to_csv(out_dir / "03_excluded_rows.csv", index=False)

print(f"-> X: {len(matrix_X)} dòng | loại: {len(excluded)} dòng (03_excluded_rows.csv)")

Đang trích xuất features cho 18997 dòng...
-> Đã xuất 03_registrable_domains.csv và 03_matrix_X_features.csv


In [ ]:
print("=" * 60)
print("HANDOFF CHECK")
print("=" * 60)
problems = []

# 1. Số dòng: X + dòng bị loại phải bằng đầu vào
n_in, n_x, n_ex, n_d = len(frame), len(matrix_X), len(excluded), len(domain_meta)
print(f"[rows] input={n_in} | X={n_x} + excluded={n_ex} | domain={n_d}")
if n_x + n_ex != n_in or n_d != n_in:
    problems.append("Số dòng không khớp")

# 2. ID trùng / thiếu / thừa / chồng lấn
ids_in, ids_x, ids_ex = set(frame[ID_COL]), set(matrix_X[ID_COL]), set(excluded[ID_COL])
dup_x = int(matrix_X[ID_COL].duplicated().sum())
dup_in = int(frame[ID_COL].duplicated().sum())
missing = ids_in - (ids_x | ids_ex)
extra = (ids_x | ids_ex) - ids_in
overlap = ids_x & ids_ex
print(f"[id] trùng(X)={dup_x} | trùng(input)={dup_in} | thiếu={len(missing)} "
      f"| thừa={len(extra)} | vừa trong X vừa bị loại={len(overlap)}")
if dup_x or dup_in or missing or extra or overlap:
    problems.append("ID trùng/thiếu/thừa/chồng lấn")

# 3. NaN / Inf trong X (phải bằng 0)
vals = matrix_X[FEATURE_COLS].to_numpy(dtype=float)
nan_total, inf_total = int(np.isnan(vals).sum()), int(np.isinf(vals).sum())
print(f"[values] NaN={nan_total} | Inf={inf_total}")
if nan_total or inf_total:
    problems.append("X còn NaN/Inf")

# 4. Dòng bị loại khỏi X và lý do
print("[excluded] theo error_reason:")
print(excluded["error_reason"].value_counts(dropna=False).to_string())
print(excluded.to_string(index=False))

# 5. Domain chưa xác định và số URL đủ điều kiện Group CV
dom_na = domain_meta["registrable_domain"].isna()
eligible = domain_meta[~dom_na]
print(f"[domain] chưa xác định={int(dom_na.sum())}/{n_d}")
print(f"[group-cv] đủ điều kiện={len(eligible)}/{n_d} ({len(eligible)/n_d:.2%}) "
      f"| số domain duy nhất={eligible['registrable_domain'].nunique()}")
top = eligible["registrable_domain"].value_counts().head(10)
print(f"[group-cv] top 10 domain (tổng {top.sum()} URL, {top.sum()/len(eligible):.2%}):")
print(top.to_string())
if "label" in frame.columns:
    print("[label]", frame.set_index(ID_COL).loc[eligible[ID_COL], "label"].value_counts().to_dict())

# 6. Cột hằng trong X
const_cols = [c for c in FEATURE_COLS if matrix_X[c].nunique(dropna=True) <= 1]
print(f"[const] cột hằng: {const_cols or 'không có'}")
print("-" * 60)

if problems:
    raise AssertionError("HANDOFF FAIL: " + "; ".join(problems))
print("HANDOFF OK: có thể bàn giao.")

HANDOFF CHECK
[rows] input=18997 | X=18997 | domain=18997
[id] trùng(X)=0 | trùng(input)=0 | thiếu=0 | thừa=0
[values] NaN=24 (trên 2 dòng) | Inf=0
[domain] chưa xác định=2/18997
   phân loại error_reason:
error_reason
unresolved_domain    2
[group-cv] URL đủ điều kiện=18995/18997 (99.99%) | số domain duy nhất=12277
[const] cột hằng: ['is_ip']
------------------------------------------------------------

[domain] URL chưa xác định:
                                  id                    url
74648118-c397-5d04-ab64-9c2973953892 vpbank.com.vn-uz.infor
27984223-c2e3-5068-a777-02f4becd0d10 techcombank.com-ut.xvz

[group-cv] top 10 domain (tổng 2975 URL, 15.66%):
registrable_domain
thanhhoa.gov.vn        614
haiphong.edu.vn        430
hanoi.gov.vn           378
thuathienhue.gov.vn    284
quangngai.gov.vn       259
namdinh.gov.vn         238
angiang.gov.vn         203
caobang.gov.vn         200
vinhlong.gov.vn        191
khanhhoa.gov.vn        178

[label] {'benign': 16410, 'phishing': 2585}

In [7]:
ipv4_like = frame["url"].astype(str).str.contains(
    r"^(?:[a-zA-Z][a-zA-Z0-9+.-]*://)?\d{1,3}(?:\.\d{1,3}){3}(?:[:/]|$)", regex=True
)
print("URL dạng IPv4 trong dữ liệu:", int(ipv4_like.sum()))

URL dạng IPv4 trong dữ liệu: 0


## 5. Kết luận
## Kết luận & Bàn giao
- **Parser**: Dùng `tldextract`/PSL, đã kiểm thử 12 trường hợp biên.
- **Ma trận X**: 18.997 dòng, gồm `id` + 12 features; không trùng/thiếu id, không Inf. **NaN = 24 ô trên 2 dòng** (do 2 URL giả mạo dùng TLD ngoài PSL: `.infor`, `.xzv`).
- **Domain Grouping**: 18.995/18.997 URL (99,99%) có `registrable_domain`; 12.277 domain duy nhất; top 10 domain chiếm 15,66% URL (lớn nhất `thanhhoa.gov.vn`: 614 URL).
- **Nhãn**: benign 16.410, phishing 2.585 (13,6%).
- **is_ip**: Là cột hằng (0) trên dữ liệu hiện tại.
- **Bàn giao**: Đã xuất `03_registrable_domains.csv` (cho Hưng) và `03_matrix_X_features.csv` (cho Phúc). 2 ID chứa NaN cần được Phúc loại bỏ hoặc impute trước khi đưa vào mô hình Logistic Regression.